# MARS Map Generation

This notebook demonstrates **automated map generation from satellite imagery** using the MARS model to extract infrastructure features at scale.

## What This Does:

**MARS** automatically analyzes high-resolution aerial imagery to create structured geospatial datasets of buildings, roads, and railways.

### Key Capabilities:
- 🏗️ **Building Extraction**: Detects building footprints as polygon geometries
- 🛣️ **Road Network Mapping**: Identifies road networks as line geometries  
- 🚂 **Railway Detection**: Maps railway infrastructure as line geometries
- 🌍 **Multi-Location Processing**: Process multiple areas of interest concurrently
- 📤 **GeoCatalog Publishing**: Results published as STAC items with full metadata

### Workflow Overview:
1. Define your areas of interest (AOI) - neighborhoods, cities, or regions
2. SDK queries STAC catalogs for available high-resolution imagery
3. MARS model analyzes imagery to detect and vectorize infrastructure
4. Results published to your GeoCatalog as searchable, reusable geospatial data

## Prerequisites:

**Before running this notebook:**

1. **Create virtual environment and install SDK**: Navigate to the SDK directory and run:
   ```bash
   # Create and activate virtual environment
   python -m venv venv
   venv\Scripts\activate  # Windows
   # source venv/bin/activate  # macOS/Linux
   
   # Install SDK (includes ipykernel)
   pip install -e .
   
   # Register Jupyter kernel
   python -m ipykernel install --user --name=geoai-sdk
   ```

2. **Azure Authentication**: Configure Azure CLI
   ```bash
   az login
   ```

3. **Select Kernel**: Choose "Python 3.12 (geoai-sdk)" as the notebook kernel

4. **Configure Settings**: Update cells with your model endpoint and output storage details

## 1. Setup and Imports

The SDK automatically configures clean, production-ready logging (INFO level by default).

**Logging Modes:**
- **INFO (default)**: High-level progress only - clean logs for production
- **DEBUG**: Detailed per-chip processing logs - useful for troubleshooting

**To enable debug mode:**
```python
os.environ['GEOAI_LOG_LEVEL'] = 'DEBUG'  # Before importing geoai
```

In [ ]:
import asyncio
import os
from azure.identity import DefaultAzureCredential

# Optional: Enable DEBUG logging for troubleshooting
# Uncomment to see detailed per-chip processing logs
# os.environ['GEOAI_LOG_LEVEL'] = 'DEBUG'

import geoai

print("✅ SDK loaded successfully!")
print(f"   Log level: {os.getenv('GEOAI_LOG_LEVEL', 'INFO')}")

## 2. Azure Authentication

**When you need Azure authentication:**
- 📤 Publishing results to GeoCatalog (includes blob storage for assets)
- 🔒 Using private GeoCatalog as input

In [ ]:
# Initialize Azure credential
credential = DefaultAzureCredential()

# Optional: Load storage account key from environment
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")

if storage_account_key:
    print("🔑 Using storage account key for SAS generation")
else:
    print("🔐 Using Azure AD authentication (recommended)")
    print("   Ensure you have 'Storage Blob Delegator' role")

## 3. Discover Available Models (Optional)

Check which models are available and what collections they support.

**About Collections:**
- **Planetary Computer**: Pre-defined datasets like NAIP, Landsat, Sentinel-2
  - EOOS/MARS support: **NAIP only** (high-resolution imagery)
- **Private GeoCatalog**: Your own ingested high-resolution imagery
  - Use any collection name you want

In [ ]:
# List all available models
print("📚 Available Models:\n")
for model_info in geoai.models.list():
    print(f"  🤖 {model_info['model_name']} ({model_info['name'].upper()})")
    print(f"     Collections: {', '.join(model_info['supported_collections'])}")
    print(f"     Bands: {', '.join(model_info['required_bands'])}")
    print(f"     Resolution: {model_info['preferred_resolution_meters']}m")
    print()

# Get detailed information about MARS
print("🔍 MARS Model Details:")
mars_info = geoai.models.get("mars")
print(f"   Supported collections: {mars_info['data_requirements']['supported_collections']}")
print(f"   Required bands: {mars_info['data_requirements']['required_bands']}")
print(f"   Detects: Building, Road, Railway")


## 4. Define Input Source

Configure where to get satellite imagery.

**Option 1: Public Planetary Computer** (default - easiest!)
- Just specify collection: `naip`
- No authentication needed
- NAIP: High-resolution aerial imagery (0.6m, USA)

**Option 2: Private GeoCatalog** (for your own data)
- Ingest your high-resolution imagery as STAC items
- Specify your GeoCatalog URI and collection name
- Requires Azure authentication

In [ ]:
# Option 1: Public Planetary Computer (default)
input_source = geoai.Input(collection="naip")  # Public collection from Microsoft Planetary Computer

# Option 2: Private GeoCatalog (uncomment and configure)
# input_source = geoai.Input(
#     collection="your-high-res-imagery",  # Your custom collection name
#     geocatalog_uri="https://your-company.geocatalog.com/stac",
#     credential=credential  # Azure credential required
# )

print(f"📥 Input configured: {input_source.collection}")
print(f"   Source: {input_source.geocatalog_uri}")

## 5. Define Areas of Interest (Multi-AOI)

This example processes **multiple locations** simultaneously to demonstrate multi-AOI capabilities.

**Example File: `test_miami_mars.geoparquet`**
- Contains **2 AOIs** in Miami Beach area for demonstration:
  - **miami_beach_1** - Miami Beach Coastal area
  - **miami_beach_2** - Miami Beach Inland area
- Each AOI covers urban areas with buildings, roads, and infrastructure
- Shows programmatic GeoDataFrame creation (alternative to loading from file)

**Multi-AOI Benefits:**
- 🌍 Process many locations in one run
- 📊 Efficient STAC search (SDK handles optimization)
- 📤 All results published to single collection
- ⚡ Concurrent chip processing across all AOIs

**NAIP Update Cycle:**
- NAIP imagery is collected approximately every **3 years**
- For best data availability, use datetime ranges **3+ years apart** (e.g., "2017-01-01/2020-12-31" or "2020-01-01/2023-12-31")
- Narrow date ranges may result in no imagery coverage for some AOIs

**Tips:**
- Use https://geojson.io to draw custom AOIs
- Can use GeoDataFrame, GeoParquet file, or GeoJSON
- Each AOI processed independently (failures don't affect others)
- Best for: Multiple buildings, neighborhoods, or districts

In [ ]:
# Create multi-AOI constraint using GeoDataFrame
# This demonstrates direct GeoDataFrame usage (alternative: load test_miami_mars.geoparquet)

import geopandas as gpd
from shapely.geometry import box

# Define multiple AOIs in Miami Beach area (matches test_miami_mars.geoparquet)
aois_data = {
    'id': ['miami_beach_1', 'miami_beach_2'],
    'name': ['Miami Beach - Coastal', 'Miami Beach - Inland'],
    'geometry': [
        box(-80.13773083243422, 25.78740094399644, -80.1352643098228, 25.790009223060604),
        box(-80.13187961358523, 25.807251402127395, -80.12776159322529, 25.811404605351793)
    ]
}

# Create GeoDataFrame with WGS84 CRS
gdf = gpd.GeoDataFrame(aois_data, crs="EPSG:4326")

# Create constraint with GeoDataFrame
constraint = geoai.Constraint(
    aois=gdf,  # Pass GeoDataFrame directly
    datetime="2020-01-01/2024-12-31"  # NAIP updates ~every 3 years - use 3+ year range for best coverage
)

print(f"⚙️  Constraint configured:")
print(f"   Mode: {constraint.mode}")  # Should show "multi"
print(f"   AOIs: {len(list(constraint.iter_aois()))}")
print(f"   Datetime: {constraint.datetime}")

# Preview AOIs
print(f"\n📍 Processing {len(gdf)} AOIs:")
for idx, row in gdf.iterrows():
    area_km2 = row.geometry.area * 111 * 111  # Rough conversion
    print(f"   {row['id'].upper()}: {row['name']} (~{area_km2:.3f} km²)")

# Alternative: Load from GeoParquet file (same AOIs as above)
# constraint = geoai.Constraint(
#     aois="test_miami_mars.geoparquet",
#     datetime="2020-01-01/2024-12-31"  # NAIP: Use 3+ year range
# )

# Alternative: Single AOI from bbox (uncomment to use instead)
# constraint = geoai.Constraint(
#     bbox=[-80.1353305066925, 25.782802504890395, -80.13279975500333, 25.785420545651412],
#     datetime="2020-01-01/2024-12-31"  # NAIP: Use 3+ year range
# )

# Optional: Filter by imagery resolution (GSD - Ground Sample Distance)
# NAIP example - find high-resolution imagery only:
# constraint = geoai.Constraint(
#     bbox=[-80.1353305066925, 25.782802504890395, -80.13279975500333, 25.785420545651412],
#     datetime="2020-01-01/2023-12-31",
#     filter={"gsd": {"lte": 0.5}}  # Only imagery 0.5m/pixel or better
# )

# Note: Cloud cover filters only work with satellite imagery (Sentinel, Landsat)
# NAIP is aerial imagery without cloud metadata, so cloud filters are not applicable
# For satellite imagery, combine GSD and cloud cover:
# filter={"gsd": {"lte": 10}, "eo:cloud_cover": {"lt": 5}}  # Sentinel/Landsat only

## 5a. Advanced: Custom STAC Search (Optional)

**Skip this section if you're using the simple constraint above.**

For power users who need full control over imagery selection (sorting, limiting, complex filters), you can build a custom STAC search and pass it directly to the SDK.

**When to use:**
- 🎯 Sort by date (latest first) or cloud cover (lowest first)
- 📊 Limit to N best items (testing, cost control)
- 🔍 Multi-AOI efficiency (one search for many locations)
- ⚙️ Complex CQL2 filter expressions

**How it works:**
- You build the search with `pystac_client`
- SDK uses your search results for all AOIs
- AOIs without imagery in search results are skipped with warning

In [ ]:
# OPTIONAL: Build custom STAC search for advanced control
# Uncomment to use this instead of the simple constraint above

# import pystac_client
# 
# # Build custom search with advanced options
# client = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
# 
# custom_search = client.search(
#     collections=["naip"],
#     bbox=[-80.1353305066925, 25.782802504890395, -80.13279975500333, 25.785420545651412],
#     datetime="2020-01-01/2024-12-31",
#     
#     # Advanced options:
#     query={"eo:cloud_cover": {"lt": 5}},  # Very low cloud cover (satellite only)
#     sortby=[{"field": "properties.datetime", "direction": "desc"}],  # Latest first
#     limit=10  # Only top 10 items
# )
# 
# # Replace simple constraint with advanced version
# constraint = geoai.Constraint(
#     bbox=[-80.1353305066925, 25.782802504890395, -80.13279975500333, 25.785420545651412],
#     stac_search=custom_search  # SDK will use your custom search
# )
# 
# print(f"📍 Advanced constraint configured")
# print(f"   Using custom STAC search with sorting/limiting")

# For multi-AOI: Search large area once, process many AOIs efficiently!
# search = client.search(bbox=[-80.35, 25.75, -80.10, 25.85], limit=50)  # All of Miami Beach
# constraint = geoai.Constraint(aois=gdf, stac_search=search)

## 6. Configure Output Destination

All results are published to GeoCatalog with assets stored in Azure Blob Storage.

**Required Setup:**

1. **GeoCatalog**: Your GeoCatalog endpoint URL and collection name

2. **Azure Blob Storage**: Storage account and container for chip imagery and STAC item assets
   - Create a storage account
   - Create a container (e.g., "sdk-results")
   
3. **Authentication** (choose one):
   - **Recommended**: Azure RBAC roles (no key needed)
     - Storage Blob Data Contributor
     - Storage Blob Delegator
   - **Alternative**: Set `STORAGE_ACCOUNT_KEY` environment variable
     ```bash
     export STORAGE_ACCOUNT_KEY="your_key_here"
     ```

**Optional**: Enable `save_local=True` to save GeoJSON files locally for debugging.

**Update the values below with your settings:**

In [ ]:
# Configure output destination (GeoCatalog + Blob Storage)
output = geoai.Output(
    # GeoCatalog endpoint (required)
    geocatalog_uri="https://cerademo.gsa0gkcaf5c9e6dv.uksouth.geocatalog.spatio.azure.com/",
    collection_name="mars-infer-01",
    credential=credential,
    
    # Blob storage for chip imagery and STAC assets (required)
    storage_url="https://planetarycomputeremo.blob.core.windows.net",
    blob_container="sdk-results",
    storage_account_key=os.getenv("STORAGE_ACCOUNT_KEY"),  # Optional: use Azure AD if not provided
    
    # Run identifier (optional - auto-generates UUID if not provided)
    run_id="mars-infer-01",
    
    # Local file saving (optional - for debugging only)
    save_local=True,  # Set to True to save GeoJSON files locally
    output_dir="./output"  # Only used if save_local=True
)

print(f"📤 Output configured:")
print(f"   GeoCatalog: {output.geocatalog_uri}")
print(f"   Collection: {output.collection_name}")
print(f"   Blob container: {output.blob_container}")
print(f"   Run ID: {output.run_id or 'auto-generate'}")
if output.save_local:
    print(f"   Local output: {output.output_dir}/")

## 7. Initialize MARS Model

Configure the model endpoint. Supports both **Azure ML** and **Azure AI Foundry** deployments.

**Authentication Options:**

| Method | Use Case | Setup |
|--------|----------|-------|
| **API Key/Token** | Simple, works everywhere | Get from Azure ML Studio or AI Foundry portal |
| **Azure AD** ⭐ | Production, no secrets in code | Requires role assignment (see below) |

**Required Azure Roles for Azure AD:**

- **Azure ML**: `AzureML Data Scientist` role on workspace
- **AI Foundry**: `Azure AI Developer` or `Cognitive Services User` role on project

**Assign roles:**
```bash
# Azure ML
az role assignment create --assignee user@contoso.com --role "AzureML Data Scientist" \
  --scope /subscriptions/{sub-id}/resourceGroups/{rg}/providers/Microsoft.MachineLearningServices/workspaces/{workspace}

# AI Foundry
az role assignment create --assignee user@contoso.com --role "Azure AI Developer" \
  --scope /subscriptions/{sub-id}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{ai-service}
```

**Note:** SDK automatically detects endpoint type and uses the correct OAuth scope.

In [ ]:
# ============================================================================
# Choose ONE authentication method below:
# ============================================================================

# ------------------------------
# Option 1: API Key / JWT Token
# ------------------------------
# Simplest method - works with both Azure ML and AI Foundry
# Get key from Azure ML Studio or AI Foundry portal

# Example for Azure ML (JWT token from "Consume" tab)
model = geoai.models.MARS(
    endpoint="https://mars-map-autoregressive-endpoint.eastus.inference.ml.azure.com/score",
    credential="api-key"  # Replace with actual key from Azure ML "Consume" tab
)

# Example for AI Foundry (API key from portal)
# model = geoai.models.MARS(
#     endpoint="https://your-model.eastus.inference.ai.azure.com/score",
#     credential=os.getenv("MARS_API_KEY", "your-api-key-here")
# )

# ------------------------------
# Option 2: Azure AD (Recommended for Production)
# ------------------------------
# Uses DefaultAzureCredential - no secrets in code!
# Requires role assignment (see Cell 14 for details)

# model = geoai.models.MARS(
#     endpoint="https://your-model.eastus.inference.ai.azure.com/score",
#     credential=credential  # DefaultAzureCredential from Cell 2
# )
#
# Required roles:
#   - Azure ML: "AzureML Data Scientist"
#   - AI Foundry: "Azure AI Developer" or "Cognitive Services User"

# ============================================================================

print("🤖 MARS model initialized")
print(f"   Endpoint: {model.endpoint}")
if isinstance(model.credential, str):
    print(f"   Auth: API Key/Token (length: {len(model.credential)} chars)")
else:
    print(f"   Auth: Azure AD (DefaultAzureCredential)")
    print(f"   ℹ️  Scope auto-detected based on endpoint URL")

## 8. Configure Detection Parameters

Set model-specific parameters for MARS processing.

**Parameters:**
- `chip_size`: Size of image chips in **pixels** (default: 1024, max: 2048) - SDK-internal parameter
- `stride`: Overlap between chips in **pixels** (default: 800) - SDK-internal parameter
- `categories`: Optional filter for specific features (default: all 3 categories) - sent to model endpoint

**Note:** MARS endpoint doesn't support threshold parameter. Detection confidence is managed by the model internally.

**💡 Recommended:** Default parameters are pre-optimized for accuracy and speed.

In [ ]:
# Detection parameters for MARS
params = {
    "chip_size": 1024,        # pixels (common: 2048, 1024)
    "stride": 800,           # pixels (no overlap when stride = chip_size)
    # "categories": ["Building"]  # Optional: filter specific categories
}

print(f"⚙️  Detection parameters:")
print(f"   Chip size: {params['chip_size']}px")
print(f"   Stride: {params['stride']}px")
categories_str = 'All (Building, Road, Railway)' if 'categories' not in params else ', '.join(params['categories'])
print(f"   Categories: {categories_str}")

## 8a. Validate Input (Optional but Recommended)

**Comprehensive Pre-flight Check**

Before running estimation or the full workflow, you can explicitly validate your configuration to catch issues early:

**Input Validation:**
- ✅ **Imagery availability** - Queries STAC to verify data exists for AOI
- ✅ **Required bands available** - Checks collection has RGB bands
- ✅ **Resolution compatibility** - Verifies resolution matches model requirements
- ✅ Collection compatibility with model
- ✅ Filter compatibility with collection
- ✅ Parameter validity (ranges, types)
- ✅ AOI format is readable

**Model Endpoint Validation:**
- ✅ **Endpoint reachable** - Tests model endpoint connectivity
- ✅ **Authentication works** - Validates your API key/credential (prevents 401 errors!)

**Output Validation (if output provided):**
- ✅ **GeoCatalog accessible** - Tests write permissions
- ✅ **Blob storage accessible** - Verifies storage account access

**Note:** Validation happens automatically in both `estimate()` and `run()`, so this step is optional.

**What gets checked:**

- Queries actual STAC catalog for your AOI
- Tests model endpoint authentication
- Verifies output destination access
- Reports number of imagery items found
- Returns detected resolution and available bands

In [ ]:
# Optional: Validate configuration before estimation
print("🔍 Validating input configuration...")
print("   This queries STAC to verify imagery availability")
print("   and tests model endpoint + output destination access\n")

validation = await model.validate_input(
    input=input_source,
    constraint=constraint,
    params=params,
    output=output  # Also validates model endpoint and output destination!
)

# Check validation result
if validation.is_valid:
    print("✅ Validation passed!")
    print(f"   📊 Found {validation.stac_items_count} imagery items")
    print(f"   📏 Detected resolution: {validation.detected_resolution}m/pixel")
    print(f"   🎨 Bands available: {validation.bands_found}")
    if validation.warnings:
        print("\n⚠️  Warnings:")
        for warning in validation.warnings:
            print(f"   • {warning}")
    else:
        print("   ✓ No issues found")
else:
    print("❌ Validation failed!")
    print("\nErrors:")
    for error in validation.errors:
        print(f"   • {error}")

    print("\n⚠️  Fix these errors before proceeding")    # raise Exception("Validation failed")
    # Uncomment to stop execution on validation failure

## 9. Estimate Job Scope (Optional but Recommended)

Before running the full workflow, estimate the job scope to:
- ✅ Verify imagery is available for your AOI
- ✅ Preview chip count and **estimated duration**
- ✅ Check for potential issues (missing data, oversized jobs)

**What estimation provides:**
- Queries STAC to verify data exists
- Calculates chip count based on AOI geometry
- **Estimates processing time** (includes inference + GeoCatalog publishing)
- Per-AOI breakdown for multi-AOI workflows

**Note:** MARS processing is slower than EOOS (~30s per chip vs ~3s), so duration estimates help plan accordingly.


**Includes automatic validation** - no need to run validate_input() separately if you run estimate().- Want to know how long processing will take

- Unfamiliar geographic areas

**When to use estimation:**- Large AOIs or many chips

In [ ]:
# Run estimation before full processing
print("📊 Running estimation...")

estimate = await model.estimate(
    input=input_source,
    constraint=constraint,
    params=params
)

# Print summary
print(f"\n✅ Estimation Complete:")
print(f"   Total chips: {estimate.estimated_chips}")
print(f"   STAC items: {estimate.stac_items_found}")
print(f"   Inference requests: {estimate.total_requests}")

# Display estimated duration
if hasattr(estimate, 'estimated_duration_minutes') and estimate.estimated_duration_minutes:
    print(f"   ⏱️  Estimated duration: ~{estimate.estimated_duration_minutes} minutes")

# Per-AOI breakdown (if multi-AOI)
if hasattr(estimate, 'aoi_estimates') and estimate.aoi_estimates:
    print(f"\n📍 Per-AOI Breakdown:")
    for aoi_data in estimate.aoi_estimates:
        aoi_id = aoi_data.get('aoi_id', 'unknown')
        chips = aoi_data.get('estimated_chips', 0)
        # Try different possible field names for STAC items
        items = (aoi_data.get('stac_items_found', 0) or 
                 aoi_data.get('stac_items', 0) or 
                 aoi_data.get('items_found', 0))
        print(f"   {aoi_id}: {chips} chips, {items} items")

# Simple validation
if estimate.stac_items_found == 0:
    print("\n❌ No imagery found - adjust datetime/AOI and re-run")
elif estimate.estimated_chips > 1000:
    print(f"\n⚠️  Large job ({estimate.estimated_chips} chips) - consider reducing AOIs or increasing stride")
else:
    print("\n✅ Ready to proceed!")


## 10. Run MARS Map Generation

Execute the detection workflow. This may take several minutes depending on AOI size.

**Workflow Steps:**
1. 🔒 Validation
2. 🔲 Chip creation
3. 🔍 STAC search
4. 📥 Image fetching
5. 🤖 Model inference (Building, Road, Railway detection)
6. 🔄 Result merging
7. 📤 GeoCatalog publishing

In [ ]:
# Run MARS map generation with publishing
# SDK handles all logging internally
result = await model.run(
    input=input_source,
    constraint=constraint,
    params=params,
    output=output
)

## 11. View Results

Review detection statistics, category breakdown, and output locations.

**Results include:**
- Category breakdown: Buildings, Roads, Railways
- Chip processing statistics
- Local GeoJSON file with all detections
- GeoCatalog publication link (if enabled)
- Blob storage location for chip imagery

In [ ]:
# Print results summary
print("\n" + "="*70)
print("🗺️  MARS Map Generation Results")
print("="*70)

# Check if multi-AOI or single AOI
if result.total_aois:
    # Multi-AOI: Show GeoCatalog first, then Local Output
    if result.published:
        print(f"\n☁️  GeoCatalog Published:")
        collection_url = f"{output.geocatalog_uri.rstrip('/')}/collections/{output.collection_name}"
        print(f"   Collection: {collection_url}")
        print(f"   Items published: {result.successful_aois} AOIs")
        print(f"   Blob storage: {output.storage_url}/{output.blob_container}/{output.run_id}/")
        print(f"\n   💡 View individual AOI results in the GeoCatalog collection")
    else:
        print(f"\n⚠️  GeoCatalog publishing: Not enabled or failed")
    
    # Local output (only shown if save_local=True)
    if output.save_local:
        print(f"\n💾 Local Output:")
        print(f"   Directory: {output.output_dir}/")
        print(f"   Structure:")
        print(f"      {output.output_dir}/")
        print(f"      ├── <aoi_id>/")
        print(f"      │   ├── detections.geojson  (filtered to AOI bounds)")
        print(f"      │   └── final_overlay.jpg   (visual overlay)")
    
    # Multi-AOI results statistics
    print(f"\n🌍 Multi-AOI Processing:")
    print(f"   Total AOIs: {result.total_aois}")
    print(f"   Successful: {result.successful_aois}")
    print(f"   Failed: {result.total_aois - result.successful_aois}")
    
    print(f"\n📈 Aggregate Statistics:")
    print(f"   Total chips: {result.total_chips}")
    print(f"   Successful: {result.successful_chips}")
    print(f"   Total detections: {result.detection_count}")
    
    print(f"\n🏗️  Detections by Category:")
    # Aggregate detection counts from all AOIs
    aoi_results = result._raw_result.get('results', [])
    if aoi_results:
        category_totals = {}
        for aoi_result in aoi_results:
            aoi_data = aoi_result.get('result', {})
            if 'detection_counts' in aoi_data:
                for category, count in aoi_data['detection_counts'].items():
                    category_totals[category] = category_totals.get(category, 0) + count
        
        if category_totals:
            for category, count in category_totals.items():
                icon = "🏢" if category == "Building" else "🛣️" if category == "Road" else "🚂"
                print(f"   {icon} {category}: {count}")
        else:
            print(f"   Total detections: {result.detection_count}")
    else:
        print(f"   Total detections: {result.detection_count}")
    
    print(f"\n📍 Per-AOI Breakdown:")
    aoi_results = result._raw_result.get('results', [])
    if aoi_results:
        for aoi_result in aoi_results:
            aoi_id = aoi_result.get('aoi_id', 'unknown')
            aoi_data = aoi_result.get('result', {})
            print(f"   {aoi_id.upper()}:")
            print(f"      Chips: {aoi_data.get('total_chips', 0)}")
            print(f"      Detections: {aoi_data.get('detection_count', 0)}")
            if 'detection_counts' in aoi_data:
                for cat, cnt in aoi_data['detection_counts'].items():
                    print(f"         {cat}: {cnt}")
else:
    # Single AOI results
    if result.published:
        print(f"\n☁️  GeoCatalog Published:")
        print(f"   Run ID: {result.run_id}")
        print(f"   Collection: {output.collection_name}")
        print(f"   View: {result.geocatalog_url}")
        print(f"   Blob path: {result.blob_base_path}")
    else:
        print(f"\n⚠️  GeoCatalog publishing: Not enabled or failed")
    
    # Local output (only shown if save_local=True)
    if output.save_local:
        print(f"\n💾 Local Output:")
        print(f"   Directory: {output.output_dir}/")
        print(f"   Files: detections.geojson, final_overlay.jpg")
    
    print(f"\n📈 Processing Summary:")
    print(f"   Total chips: {result.total_chips}")
    print(f"   Successful: {result.successful_chips}")
    print(f"   Failed: {result.total_chips - result.successful_chips}")
    
    print(f"\n🏗️  Detections by Category:")
    if hasattr(result, 'detection_counts') and result.detection_counts:
        for category, count in result.detection_counts.items():
            icon = "🏢" if category == "Building" else "🛣️" if category == "Road" else "🚂"
            print(f"   {icon} {category}: {count}")
    else:
        print(f"   Total detections: {result.detection_count}")

print("\n" + "="*70)

## 12. Next Steps

**View Results:**
- Open GeoCatalog URL above to visualize detections
  - Buildings: Red polygons
  - Roads: Cyan lines
  - Railways: Yellow lines
- Check `./output/` directory for local files and visualizations
- Load GeoJSON in QGIS, ArcGIS, or other GIS tools

**Process More Locations:**
- Add more AOIs to the GeoDataFrame in section 5
- Use GeoParquet file like EOOS example: `constraint = geoai.Constraint(aois="areas.geoparquet", ...)`
- Create custom AOIs with different Miami neighborhoods

**Adjust Parameters:**
- **Threshold** (section 8): Increase to reduce false positives (try 0.7-0.8)
- **Categories** (section 8): Filter specific features `["Building"]` or `["Road", "Railway"]`
- **Chip size** (section 8): Larger chips (1024px) for coverage, smaller (512px) for detail
- **AOIs** (section 5): Add/remove locations, change datetime range

**Before Large Jobs:**
- Always run estimation (section 9) to verify data availability
- Check chip count and adjust stride if needed
- Test on small AOI subset first to validate parameters

**Multi-AOI Tips:**
- SDK processes each AOI independently
- Failed AOIs don't stop the entire job
- All results published to single GeoCatalog collection
- Check per-AOI breakdown in results

**Troubleshooting:**
- **No detections**: Lower threshold, verify AOI has visible structures
- **Too many false positives**: Increase threshold, filter by category
- **Missing buildings**: Check AOI has clear building footprints
- **Authentication errors**: Run `az login` and verify RBAC roles
- **GeoCatalog errors**: Check collection exists and credentials have write access
- **Some AOIs skipped**: Check STAC item coverage, expand datetime range

---

## 📚 Additional Resources

**Documentation:**
- [SDK Documentation](../../README.md) - Complete SDK reference
- [Examples README](../README.md) - All example notebooks
- [EOOS Notebook](../EO_OS_Object_Detection/eoos_object_detection.ipynb) - Object detection workflow

**External Resources:**
- [MARS Model Paper](https://arxiv.org/abs/2212.07689) - Research paper
- [Planetary Computer](https://planetarycomputer.microsoft.com/) - Public STAC catalog
- [NAIP Imagery](https://planetarycomputer.microsoft.com/dataset/naip) - Dataset documentation
- [STAC Specification](https://stacspec.org/) - STAC API reference

**SDK Features:**
- 🔍 **Discovery**: `models.list()`, `models.get()`, `Model.get_supported_collections()`
- 🔒 **Validation**: Automatic input/collection validation before processing
- 📊 **Estimation**: `model.estimate()` - Preview chip count and verify data
- 🚀 **Execution**: `model.run()` - Process with automatic validation

**Model Details:**
- MARS detects: Building (Polygon), Road (LineString), Railway (LineString)
- Optimized for high-resolution aerial imagery (< 1m/pixel)
- Returns native GeoJSON geometries (not just bounding boxes)

**Support:**
- GitHub: Report issues or contribute
- Azure Orbital Spatio Team: Technical support